# CIS 635 — Knowledge Discovery & Data Mining
## Introduction to KDD and Python Basics

### Interactive Google Colaboratory Notebook

This notebook is an interactive companion to the course reading **“Introduction to Knowledge Discovery and Data Mining, and Python Basics.”** It follows the reading's organization and emphasizes both the **KDD process** and the **plain-Python foundations** needed to implement its early stages.

### Learning objectives
By completing this notebook, you should be able to:

- Distinguish KDD from data mining and related fields.
- Describe the five stages of the classic KDD process.
- Compare the classic KDD process with CRISP-DM.
- Explain why KDD is iterative rather than a one-way pipeline.
- Trace a small, messy dataset through Selection, Preprocessing, Transformation, Data Mining, and Interpretation/Evaluation.
- Use Python variables, data types, operators, control flow, functions, list comprehensions, and built-in data structures.
- Apply these concepts to realistic KDD examples.

> **How to use this notebook:** Read each section, run every code cell, modify examples, and complete the exercises. Every executable code cell is preceded by a short explanation of its purpose.

# 1. KDD: From Raw Data to Knowledge

**Knowledge Discovery in Databases (KDD)** is the end-to-end process of turning data into useful knowledge. In the terminology used in the reading, **data mining** is one stage within the larger KDD process.

A useful distinction:

| Area | Primary concern |
|---|---|
| Statistics | Quantifying uncertainty and testing hypotheses |
| Databases | Storing, indexing, and querying data efficiently |
| Machine Learning | Learning patterns or models from data |
| Data Mining | Applying methods to extract candidate patterns |
| KDD | Selecting, cleaning, transforming, mining, and interpreting data |
| AI | The broader field of building systems that perform intelligent tasks |

## Quick reflection
Before continuing, describe one example from a domain you know where raw records could potentially be turned into useful knowledge.

# 2. The Classic KDD Process

The five classic stages are:

1. **Selection** — choose the data relevant to the question.
2. **Preprocessing** — address missing values, duplicates, errors, inconsistent formats, and other quality problems.
3. **Transformation** — reshape or derive features suitable for analysis.
4. **Data Mining** — apply a technique to extract candidate patterns.
5. **Interpretation and Evaluation** — determine whether a pattern is valid, useful, and worth acting on.

The process is **iterative**. A poor result at a later stage may require revisiting an earlier stage.

**Code cell purpose:** Create a simple list representing the five classic KDD stages and print them in order.

In [ ]:
kdd_stages = [
    "Selection",
    "Preprocessing",
    "Transformation",
    "Data Mining",
    "Interpretation and Evaluation",
]

for i, stage in enumerate(kdd_stages, start=1):
    print(f"{i}. {stage}")

# 3. CRISP-DM and Its Relationship to KDD

CRISP-DM organizes similar work into six phases:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

A useful conceptual mapping is:

- Business Understanding → implicit before KDD Selection
- Data Understanding → related to Selection and early exploration
- Data Preparation → Preprocessing + Transformation
- Modeling → Data Mining
- Evaluation → Interpretation and Evaluation
- Deployment → explicit post-evaluation operational use

Both models should be treated as **cyclical rather than strictly linear**.

**Code cell purpose:** Store the CRISP-DM phases and a simple mapping to the nearest classic KDD stage.

In [ ]:
crisp_dm_mapping = {
    "Business Understanding": "Before / guides Selection",
    "Data Understanding": "Selection and early exploration",
    "Data Preparation": "Preprocessing + Transformation",
    "Modeling": "Data Mining",
    "Evaluation": "Interpretation and Evaluation",
    "Deployment": "After evaluation; operational use and monitoring",
}

for phase, mapping in crisp_dm_mapping.items():
    print(f"{phase:25s} -> {mapping}")

## Exercise 1 — KDD Concepts

Answer in your own words:

1. What is the difference between **KDD** and **data mining**?
2. Give one example of a KDD activity that is not the data-mining stage.
3. Describe a situation in which a team would need to return from Data Mining to Preprocessing or Transformation.
4. Why does CRISP-DM make Business Understanding and Deployment explicit?

# 4. Worked Example: A Messy Farm-Sensor Dataset

The following example follows the course reading's scenario: a farm cooperative wants to investigate whether **soil moisture relates to crop yield**. The raw data intentionally includes:

- a missing soil-moisture value (`None`)
- an invalid sensor fault code (`-999.0`)

We will trace this dataset through all five KDD stages using only plain Python.

**Code cell purpose:** Create the raw farm-sensor dataset used throughout the KDD walkthrough.

In [ ]:
raw_log = [
    {"field_id": "A1", "sensor_batch": 7, "soil_moisture_pct": 22.4,
     "rainfall_mm": 4.0, "temperature_c": 24.1, "yield_kg_per_ha": 3120},
    {"field_id": "A2", "sensor_batch": 7, "soil_moisture_pct": 18.1,
     "rainfall_mm": 1.2, "temperature_c": 26.5, "yield_kg_per_ha": 2870},
    {"field_id": "A3", "sensor_batch": 7, "soil_moisture_pct": None,
     "rainfall_mm": 0.0, "temperature_c": 27.8, "yield_kg_per_ha": 2510},
    {"field_id": "B1", "sensor_batch": 8, "soil_moisture_pct": 31.6,
     "rainfall_mm": 12.4, "temperature_c": 21.9, "yield_kg_per_ha": 3810},
    {"field_id": "B2", "sensor_batch": 8, "soil_moisture_pct": 29.0,
     "rainfall_mm": 9.8, "temperature_c": 22.4, "yield_kg_per_ha": 3655},
    {"field_id": "B3", "sensor_batch": 8, "soil_moisture_pct": -999.0,
     "rainfall_mm": 10.1, "temperature_c": 22.0, "yield_kg_per_ha": 3702},
    {"field_id": "C1", "sensor_batch": 9, "soil_moisture_pct": 15.2,
     "rainfall_mm": 0.4, "temperature_c": 29.3, "yield_kg_per_ha": 2255},
    {"field_id": "C2", "sensor_batch": 9, "soil_moisture_pct": 16.8,
     "rainfall_mm": 0.6, "temperature_c": 28.9, "yield_kg_per_ha": 2390},
]

print("Number of raw records:", len(raw_log))
for row in raw_log:
    print(row)

## Stage 1: Selection

Keep only the fields needed for the current question: field ID, soil moisture, and yield.

**Code cell purpose:** Define a selection function and use a list comprehension to create a target dataset.

In [ ]:
def select_fields(row):
    return {
        "field_id": row["field_id"],
        "soil_moisture_pct": row["soil_moisture_pct"],
        "yield_kg_per_ha": row["yield_kg_per_ha"],
    }

selected = [select_fields(row) for row in raw_log]

print("Selected records:")
for row in selected:
    print(row)

## Stage 2: Preprocessing

Remove records with missing or invalid soil-moisture values. Here, a robust basic rule treats missing values and negative readings as invalid.

**Code cell purpose:** Filter the selected records using a list comprehension and report how many records were removed.

In [ ]:
cleaned = [
    row for row in selected
    if row["soil_moisture_pct"] is not None
    and row["soil_moisture_pct"] >= 0
]

removed = len(selected) - len(cleaned)
print(f"Removed {removed} record(s) with missing or invalid readings.")

for row in cleaned:
    print(row)

## Stage 3: Transformation

Derive a categorical feature, `moisture_category`, from the numeric soil-moisture value.

**Code cell purpose:** Create a transformation function and add the derived category to each cleaned record.

In [ ]:
def categorize_moisture(pct):
    return "dry" if pct < 20.0 else "moist"

transformed = [
    {**row, "moisture_category": categorize_moisture(row["soil_moisture_pct"])}
    for row in cleaned
]

for row in transformed:
    print(row)

## Stage 4: Data Mining

Use a simple group comparison: compute average crop yield for dry and moist fields.

**Code cell purpose:** Define a small average function, split yields by category, and calculate the group averages.

In [ ]:
def average(values):
    return sum(values) / len(values)

dry_yields = [
    r["yield_kg_per_ha"]
    for r in transformed
    if r["moisture_category"] == "dry"
]

moist_yields = [
    r["yield_kg_per_ha"]
    for r in transformed
    if r["moisture_category"] == "moist"
]

dry_average = average(dry_yields)
moist_average = average(moist_yields)

print(f"Average yield, dry fields:   {dry_average:.1f} kg/ha (n={len(dry_yields)})")
print(f"Average yield, moist fields: {moist_average:.1f} kg/ha (n={len(moist_yields)})")

## Stage 5: Interpretation and Evaluation

A numeric difference is not automatically a validated or actionable conclusion. We should state the observed pattern and also its limitations.

**Code cell purpose:** Calculate the observed difference and print caveats consistent with the small sample.

In [ ]:
difference = moist_average - dry_average

print(f"Moist fields out-yielded dry fields by {difference:.1f} kg/ha in this sample.")
print()
print("Interpretation caveats:")
print("- The cleaned sample is very small.")
print("- Rainfall and temperature were excluded from this simplified analysis.")
print("- No statistical test has been performed.")
print("- The pattern is worth further investigation, not yet a validated causal conclusion.")

## Exercise 2 — KDD Worked Example

1. Modify `categorize_moisture()` to use three categories: `dry`, `moderate`, and `moist`.
2. Revisit Selection and include `rainfall_mm`. Explain why returning to an earlier stage may be necessary.
3. Compare a robust preprocessing rule such as `>= 0` with a fragile rule that removes only `-999.0`.
4. Construct one additional invalid record and verify that your preprocessing rule handles it.
5. In 3–5 sentences, explain why the final pattern is not automatically proof that soil moisture causes the yield difference.

**Code cell purpose:** Use this workspace to implement your solution to Exercise 2.

In [ ]:
# Write your Exercise 2 solution here.
# You may copy and modify the earlier farm-sensor pipeline.

# 5. Real-World Applications and Evidence

The reading discusses KDD applications in retail, fraud detection, healthcare, agriculture, education, and environmental monitoring.

A key lesson is that a memorable anecdote is not automatically a reliable case study. During Interpretation and Evaluation, evidence quality matters: ask what was measured, on what data, and whether the original source can be checked.

## Reflection
Choose one application domain and identify:

- the likely raw data,
- one data-quality problem,
- one possible transformed feature,
- one pattern-mining or modeling task,
- one evaluation question a domain expert should ask.

# 6. Python Basics: Variables and Data Types

The built-in types emphasized in the reading are:

- `int` — whole numbers
- `float` — decimal numbers
- `str` — text
- `bool` — `True` or `False`
- `None` — the deliberate absence of a recorded value

For KDD, it is especially important not to confuse `None` with a real value such as `0`.

**Code cell purpose:** Create variables representing a simple patient record and inspect their values and Python data types.

In [ ]:
patient_id = "P-10432"
age = 47
body_temp_c = 38.6
is_fever = body_temp_c > 37.5
allergy_notes = None

for name, value in [
    ("patient_id", patient_id),
    ("age", age),
    ("body_temp_c", body_temp_c),
    ("is_fever", is_fever),
    ("allergy_notes", allergy_notes),
]:
    print(f"{name} = {value!r} -> type {type(value).__name__}")

# 7. Operators

**Code cell purpose:** Demonstrate arithmetic operators using a simple healthcare billing example.

In [ ]:
visit_fee = 120.00
lab_fee = 45.50
discount_pct = 10

subtotal = visit_fee + lab_fee
discount_amount = subtotal * discount_pct / 100
total_due = subtotal - discount_amount

print("Subtotal:", subtotal)
print("Discount amount:", discount_amount)
print("Total due:", total_due)
print("47 // 10:", 47 // 10)
print("47 % 10:", 47 % 10)
print("2 ** 5:", 2 ** 5)

**Code cell purpose:** Demonstrate comparison, logical, and assignment operators used in filtering and rule construction.

In [ ]:
temperature_threshold = 37.5
age_threshold = 65

is_high_risk = (body_temp_c > temperature_threshold) and (age >= age_threshold)
is_worth_flagging = (body_temp_c > temperature_threshold) or (age >= age_threshold)

running_total = 0
running_total += subtotal

print("High risk:", is_high_risk)
print("Worth flagging:", is_worth_flagging)
print("Not fever:", not is_fever)
print("Age != 47:", age != 47)
print("Running total:", running_total)

## Exercise 3 — Variables and Operators

1. Create `item_price`, `quantity`, and `is_member_discount_applied`.
2. Compute a final total with a 5% discount only when the Boolean is `True`.
3. Compare `82` and `82.0` using both `==` and `type()`.
4. Write a compound Boolean expression that flags a sensor when temperature is below `-10`, above `45`, or humidity is `None`.

**Code cell purpose:** Use this workspace to solve Exercise 3.

In [ ]:
# Write your Exercise 3 solution here.

# 8. Control Flow: Conditional Statements and Loops

**Code cell purpose:** Define a letter-grade function using an if/elif/else chain and test several scores.

In [ ]:
def letter_grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"

for score in [95, 82, 71, 58]:
    print(f"Score {score} -> Grade {letter_grade(score)}")

**Code cell purpose:** Use a for loop to process each score in a sequence and then use enumerate to include positions.

In [ ]:
scores = [95, 82, 71, 58, 88, 64, 90]

print("Scores and grades:")
for score in scores:
    print(f"{score} -> {letter_grade(score)}")

print("\nWith positions:")
for i, score in enumerate(scores, start=1):
    print(f"Student #{i}: score {score}")

**Code cell purpose:** Use a bounded while loop to process sensor readings until three consecutive values fall within the safe range.

In [ ]:
sensor_values = [45.2, 56.0, 92.4, 46.6, 50.8, 58.7]

consecutive_stable = 0
attempts = 0
readings = []

while consecutive_stable < 3 and attempts < len(sensor_values):
    value = sensor_values[attempts]
    readings.append(value)
    attempts += 1

    if 30.0 <= value <= 70.0:
        consecutive_stable += 1
    else:
        consecutive_stable = 0

print("Readings taken:", readings)
print("Attempts used:", attempts)
print("Stopped after 3 consecutive stable readings:", consecutive_stable == 3)

**Code cell purpose:** Use list comprehensions for transformation and filtering, patterns that recur in KDD Selection, Preprocessing, and Transformation.

In [ ]:
grades = [letter_grade(s) for s in scores]
passing_scores = [s for s in scores if s >= 60]
curved_passing = [s + 5 for s in scores if s >= 60]

print("Grades:", grades)
print("Passing scores:", passing_scores)
print("Curved passing scores:", curved_passing)

## Exercise 4 — Control Flow

1. Write `risk_category(days_since_last_login)` returning `low`, `medium`, or `high`.
2. Count letter grades from a list of at least ten scores using a `for` loop and dictionary.
3. Modify the `while` example to stop after three consecutive unsafe readings.
4. Convert a simple filtering loop into a list comprehension.
5. Extract all `field_id` values categorized as `moist` from the farm dataset.

**Code cell purpose:** Use this workspace to solve Exercise 4.

In [ ]:
# Write your Exercise 4 solution here.

# 9. Functions

**Code cell purpose:** Define and call a simple conversion function with a docstring and return value.

In [ ]:
def celsius_to_fahrenheit(temp_c):
    """Convert a Celsius temperature to Fahrenheit."""
    return temp_c * 9 / 5 + 32

for temp_c in [0, 20, 37, 100]:
    print(f"{temp_c} C -> {celsius_to_fahrenheit(temp_c)} F")

**Code cell purpose:** Define a function with multiple parameters and a default argument.

In [ ]:
def loan_monthly_payment(principal, annual_rate_pct, years, periods_per_year=12):
    """Compute a fixed payment for a standard amortizing loan."""
    n_periods = years * periods_per_year
    rate_per_period = (annual_rate_pct / 100) / periods_per_year

    if rate_per_period == 0:
        return principal / n_periods

    payment = principal * (
        rate_per_period * (1 + rate_per_period) ** n_periods
    ) / ((1 + rate_per_period) ** n_periods - 1)

    return payment

print("Monthly:", round(loan_monthly_payment(20000, 6.0, 5), 2))
print("Quarterly:", round(loan_monthly_payment(20000, 6.0, 5, periods_per_year=4), 2))

**Code cell purpose:** Define a function that returns multiple values and unpack the returned tuple.

In [ ]:
def summarize(values):
    """Return the minimum, maximum, and mean of a list of numbers."""
    total = 0
    lowest = values[0]
    highest = values[0]

    for v in values:
        total += v
        if v < lowest:
            lowest = v
        if v > highest:
            highest = v

    mean = total / len(values)
    return lowest, highest, mean

class_sizes = [22, 31, 18, 27, 25, 40, 19]
low, high, mean = summarize(class_sizes)

print("Minimum:", low)
print("Maximum:", high)
print("Mean:", mean)

**Code cell purpose:** Demonstrate function composition by reusing summarize inside another function.

In [ ]:
def flag_oversized_classes(class_sizes, cap=30):
    _, largest, _ = summarize(class_sizes)
    oversized = [size for size in class_sizes if size > cap]
    over_by = largest - cap if largest > cap else 0
    return oversized, over_by

oversized, over_by = flag_oversized_classes(class_sizes, cap=30)
print("Oversized classes:", oversized)
print("Largest class exceeds cap by:", over_by)

## Exercise 5 — Functions

1. Write `fahrenheit_to_celsius(temp_f)` and test a round trip from Celsius to Fahrenheit and back.
2. Write a function that determines whether a bookstore order should be kept during Preprocessing.
3. Write `summarize_by_category(records, category_key, value_key)` using only loops and dictionaries.
4. Explain why a named validation function can improve readability even if it is used only once.

**Code cell purpose:** Use this workspace to solve Exercise 5.

In [ ]:
# Write your Exercise 5 solution here.

# 10. Built-In Data Structures

**Code cell purpose:** Demonstrate list indexing, negative indexing, slicing, appending, length, and sorting.

In [ ]:
daily_sales = [412.50, 389.00, 455.75, 402.10, 610.25, 720.00, 388.90]

print("First:", daily_sales[0])
print("Last:", daily_sales[-1])
print("Friday-Saturday:", daily_sales[4:6])

daily_sales.append(399.40)
print("After append:", daily_sales)
print("Length:", len(daily_sales))
print("Sorted:", sorted(daily_sales))

**Code cell purpose:** Create a tuple representing a fixed coordinate pair and unpack it.

In [ ]:
station_location = (39.1031, -84.5120)
lat, lon = station_location

print("Station location:", station_location)
print("Latitude:", lat)
print("Longitude:", lon)

**Code cell purpose:** Use a dictionary to represent a record, update a value, add a field, and safely retrieve a missing field.

In [ ]:
patient_record = {
    "patient_id": "P-10432",
    "age": 47,
    "diagnosis_code": "J45.909",
    "visits_this_year": 3,
}

patient_record["visits_this_year"] += 1
patient_record["last_visit_flagged"] = True

print("Patient record:", patient_record)
print("Allergy:", patient_record.get("allergy", "not recorded"))
print("Keys:", list(patient_record.keys()))

**Code cell purpose:** Demonstrate set operations and de-duplication.

In [ ]:
section_a_topics = {"loops", "functions", "lists", "dictionaries"}
section_b_topics = {"functions", "dictionaries", "sets", "recursion"}

print("Intersection:", section_a_topics & section_b_topics)
print("A only:", section_a_topics - section_b_topics)
print("Exactly one:", section_a_topics ^ section_b_topics)

raw_field_ids = ["A1", "A2", "A3", "A1", "B1", "A2", "B1", "C1"]
unique_field_ids = set(raw_field_ids)

print("\nOriginal count:", len(raw_field_ids))
print("Unique count:", len(unique_field_ids))
print("Duplicates removed:", len(raw_field_ids) - len(unique_field_ids))
print("Sorted unique IDs:", sorted(unique_field_ids))

## Exercise 6 — Data Structures

1. Use list indexing and slicing to extract the first three and last two weekly temperatures.
2. Explain when a tuple is more appropriate than a list.
3. Use a dictionary accumulator to count grades.
4. Use set operations to compare diagnosis codes from two hospital wards.
5. Remove duplicate emails with a set and sort the final result for reproducible output.

**Code cell purpose:** Use this workspace to solve Exercise 6.

In [ ]:
# Write your Exercise 6 solution here.

# 11. Capstone: End-to-End KDD Walkthrough

The capstone uses a small online bookstore. The business question is:

> **Which genre generates the most revenue?**

The pipeline will:

1. Select relevant fields.
2. Remove invalid records.
3. Derive `order_total` and `revenue_tier`.
4. Aggregate revenue and order counts by genre.
5. Interpret the result with appropriate caveats.

**Code cell purpose:** Create the raw bookstore order log, including one missing genre and one invalid negative price.

In [ ]:
raw_orders = [
    {"order_id": "O1001", "customer": "Aiko", "genre": "mystery", "price": 14.99, "qty": 1, "country": "Japan"},
    {"order_id": "O1002", "customer": "Bilal", "genre": "sci-fi", "price": 9.50, "qty": 2, "country": "UAE"},
    {"order_id": "O1003", "customer": "Chen", "genre": "mystery", "price": 14.99, "qty": 1, "country": "China"},
    {"order_id": "O1004", "customer": "Dana", "genre": None, "price": 22.00, "qty": 1, "country": "USA"},
    {"order_id": "O1005", "customer": "Aiko", "genre": "sci-fi", "price": 9.50, "qty": 3, "country": "Japan"},
    {"order_id": "O1006", "customer": "Elin", "genre": "romance", "price": -5.00, "qty": 1, "country": "Sweden"},
    {"order_id": "O1007", "customer": "Farid", "genre": "mystery", "price": 14.99, "qty": 2, "country": "Egypt"},
    {"order_id": "O1008", "customer": "Grace", "genre": "romance", "price": 11.25, "qty": 1, "country": "USA"},
    {"order_id": "O1009", "customer": "Chen", "genre": "sci-fi", "price": 9.50, "qty": 1, "country": "China"},
    {"order_id": "O1010", "customer": "Hana", "genre": "romance", "price": 11.25, "qty": 2, "country": "Egypt"},
]

print("Raw orders:", len(raw_orders))

**Code cell purpose:** Implement the Selection, Preprocessing, and Transformation stages as clearly named functions and comprehensions.

In [ ]:
def select_order_fields(order):
    return {
        "order_id": order["order_id"],
        "genre": order["genre"],
        "price": order["price"],
        "qty": order["qty"],
        "country": order["country"],
    }

def is_valid_order(order):
    return order["genre"] is not None and order["price"] >= 0 and order["qty"] > 0

def revenue_tier(order_total):
    if order_total < 20:
        return "low"
    elif order_total < 50:
        return "medium"
    else:
        return "high"

selected_orders = [select_order_fields(order) for order in raw_orders]
cleaned_orders = [order for order in selected_orders if is_valid_order(order)]
transformed_orders = [
    {
        **order,
        "order_total": order["price"] * order["qty"],
        "revenue_tier": revenue_tier(order["price"] * order["qty"]),
    }
    for order in cleaned_orders
]

print("Selected:", len(selected_orders))
print("Cleaned:", len(cleaned_orders))
print("Transformed:")
for order in transformed_orders:
    print(order)

**Code cell purpose:** Aggregate total revenue, order counts, unit sales, and distinct countries by genre.

In [ ]:
revenue_by_genre = {}
count_by_genre = {}
units_by_genre = {}
countries_by_genre = {}

for order in transformed_orders:
    genre = order["genre"]

    revenue_by_genre[genre] = (
        revenue_by_genre.get(genre, 0) + order["order_total"]
    )
    count_by_genre[genre] = count_by_genre.get(genre, 0) + 1
    units_by_genre[genre] = units_by_genre.get(genre, 0) + order["qty"]

    if genre not in countries_by_genre:
        countries_by_genre[genre] = set()
    countries_by_genre[genre].add(order["country"])

for genre in sorted(revenue_by_genre):
    avg_order = revenue_by_genre[genre] / count_by_genre[genre]
    print(
        f"{genre:10s}: {count_by_genre[genre]} order(s), "
        f"revenue ${revenue_by_genre[genre]:.2f}, "
        f"avg ${avg_order:.2f}/order, "
        f"units {units_by_genre[genre]}"
    )

print("\nDistinct countries per genre:")
for genre in sorted(countries_by_genre):
    print(f"{genre:10s}: {sorted(countries_by_genre[genre])}")

**Code cell purpose:** Identify the highest-revenue genre and print an interpretation with sample-size caveats.

In [ ]:
top_genre = max(revenue_by_genre, key=revenue_by_genre.get)

print(
    f"'{top_genre}' generated the most total revenue "
    f"(${revenue_by_genre[top_genre]:.2f}) in this sample."
)

print("\nInterpretation caveats:")
print("- Only 10 raw orders were available.")
print("- Two records were removed during preprocessing.")
print("- The result may change with additional orders.")
print("- Revenue leadership does not necessarily imply broad popularity.")
print("- A business decision should consider more data and how the sample was collected.")

# 12. Capstone Exercises

### Exercise 7 — Rebuild the Pipeline
Without copying the earlier functions verbatim:

1. Write your own `select_fields`.
2. Write your own validity-check function.
3. Create a revenue tier.
4. Reproduce revenue and order counts by genre.

### Exercise 8 — Units vs. Revenue
Extend the analysis to compare:

- total revenue by genre
- total units sold by genre

Is the highest-revenue genre also the highest-volume genre? What might a discrepancy mean?

### Exercise 9 — Alternative Preprocessing
Instead of dropping records with a missing genre, replace the missing genre with `"unknown"`. Re-run the pipeline and explain one benefit and one risk of this policy.

### Exercise 10 — Add New Data
Add two orders in a new genre such as `"fantasy"`. Re-run the analysis and examine whether the leading genre changes.

### Exercise 11 — Interpretation
Write a 5–8 sentence comparison of the farm-sensor and bookstore examples. What stayed the same about the KDD process, and what changed because of the data and question?

**Code cell purpose:** Use this workspace to complete the capstone exercises.

In [ ]:
# Write your capstone exercise solution here.

# 13. Best Practices and Common Pitfalls

Keep these principles in mind:

- **Do not skip KDD stages.**
- **Treat `None` as unknown, not automatically as zero or false.**
- **Choose lists, tuples, dictionaries, and sets deliberately.**
- **A discovered pattern is not yet knowledge until it is interpreted and evaluated.**
- **Be skeptical of famous anecdotes and unsupported claims.**
- **Name functions after what they do, not how they are implemented.**
- **Do not rely on set iteration order; sort when reproducible output matters.**

# Final Self-Check

Before moving on, verify that you can independently:

1. Explain the difference between KDD and data mining.
2. List and describe all five classic KDD stages.
3. Explain why KDD is iterative.
4. Compare the emphasis of KDD and CRISP-DM.
5. Clean a small dataset containing `None` and invalid numeric values.
6. Use a list comprehension for filtering or transformation.
7. Write and call a function with parameters and return values.
8. Choose between a list, tuple, dictionary, and set for a data-representation task.
9. Build a small end-to-end KDD pipeline using only plain Python.
10. State appropriate limitations before turning a numerical pattern into a decision.